In [1]:
import duckdb
import polars as pl
import pandas as pd
import plotly.express as px
import seaborn as sns
import matplotlib.pyplot as plt

con = duckdb.connect()
PARQUET_ROOT = "/Volumes/SN7100-2TB/parquet"

plt.style.use("default")
pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 200)

print("Environment ready")

Environment ready


In [7]:
# Cell 2: Define dataset paths
bank_accounts_path = f"{PARQUET_ROOT}/bankaccounts/**/*.parquet"
inappreviews_path = f"{PARQUET_ROOT}/inappreviews/**/*.parquet"

print(bank_accounts_path)
print(inappreviews_path)


/Volumes/SN7100-2TB/parquet/bankaccounts/**/*.parquet
/Volumes/SN7100-2TB/parquet/inappreviews/**/*.parquet


In [11]:
# Extract users who requested personal finance management tools
reviews_df = pd.read_parquet(f"{PARQUET_ROOT}/inappreviews")
pfm_users = reviews_df[reviews_df['productRecommendations'].apply(lambda x: 'Personal finance management tools' in x)]
pfm_users.head()

,_id,userId,rating,issues,productRecommendations,createdAt,updatedAt,__v
15,69c0d2ee6d617c3f387e0925,6955f43d6e02ce0cdb9e8e1b,NaN,[],"[Savings account, Investing, Personal finance ...",2026-03-23 05:43:10.094,2026-03-23 05:43:10.094,0
20,69c0db1bec94e572facde9a4,69c0b3e3ec94e572fac5ffe3,NaN,[],"[Savings account, Personal finance management ...",2026-03-23 06:18:03.489,2026-03-23 06:18:03.489,0
36,69c0ffdb6d617c3f38853484,69af0428a1757267c9f90343,NaN,[],[Personal finance management tools],2026-03-23 08:54:51.252,2026-03-23 08:54:51.252,0
48,69c114c06d617c3f388892ba,691880bdbb11ef94e432ff0c,NaN,[],"[Savings account, Investing, Personal finance ...",2026-03-23 10:24:00.236,2026-03-23 10:24:00.236,0
77,69c13ede6d617c3f389253ae,6913edb5d8051a5d7517d240,NaN,[],[Personal finance management tools],2026-03-23 13:23:42.767,2026-03-23 13:23:42.767,0


In [12]:
# Get bank accounts and daily balances for PFM users
pfm_user_ids = pfm_users['userId'].unique()

bank_accounts_df = pd.read_parquet(f"{PARQUET_ROOT}/bankaccounts")
daily_balances_df = pd.read_parquet(f"{PARQUET_ROOT}/dailybankaccountbalances")

# Filter to PFM users
pfm_bank_accounts = bank_accounts_df[bank_accounts_df['userId'].isin(pfm_user_ids)]
pfm_daily_balances = daily_balances_df[daily_balances_df['userId'].isin(pfm_user_ids)]

print(f"PFM users: {len(pfm_user_ids)}")
print(f"Their bank accounts: {len(pfm_bank_accounts)}")
print(f"Their daily balances: {len(pfm_daily_balances)}")
print()
print("=== Bank Accounts Sample ===")
print(pfm_bank_accounts.head())
print()
print("=== Daily Balances Sample ===")
print(pfm_daily_balances.head())

PFM users: 118
Their bank accounts: 328
Their daily balances: 12048

=== Bank Accounts Sample ===
                            _id                    userId        type  \
17934  677305f75f84dc45f0000bc1  676ed74377247da7dfe03472  DEPOSITORY   
17935  677305f95f84dc45f0000bcd  676ed74377247da7dfe03472  DEPOSITORY   
20581  67b89f41d95483d556371ce2  67b89e3c47e9b503ef1e30cf  DEPOSITORY   
68254  68f56889ae563b7c9995b65a  68f5682987b091d6eab6819c  DEPOSITORY   
78811  68ff8731098640bfda9851c1  68ff862b9080f955a7d31042  DEPOSITORY   

        subtype                  name connectionStatus  isPrimary  \
17934  CHECKING         Free Checking        connected       True   
17935   SAVINGS         Share Savings        connected      False   
20581  CHECKING  CHASE SECURE BANKING        connected       True   
68254  CHECKING     Varo Bank Account        connected       True   
78811  CHECKING              CHECKING        connected      False   

                    createdAt isNonDebitable    